# AeroSense AI V2 — Scientifically Validated GPU Training & Spatial Deep Learning

**Platform**: Google Colab (with GPU acceleration: T4 / V100 / A100)  
**Project**: AeroSense AI V2 — Multi-City Air Quality Forecasting Platform  
**Version**: 2.0.0-PROD Readiness  

---
### Purpose & Protocol
This notebook provides a **reproducible, leak-free, scientifically rigorous** training workflow for:
1. **Experiment A (Beijing)**: ST-CNN-GRU with genuine 4D spatiotemporal tensors $(B, T=24, N=12, F=63)$ and inter-station Haversine spatial adjacency.
2. **Experiment B (Delhi NCR)**: Independent ST-CNN-GRU with Delhi's 12-station network and city-specific Haversine graph adjacency matrix (NOT reusing Beijing coordinates).
3. **Experiment C (Mumbai)**: Spatial feasibility analysis on Mumbai's 8 stations ($8\times 8$ graph) and transfer fine-tuning assessment.
4. **Experiment D (Vijayawada)**: Small-data validation on 2 stations (proving why forcing a 12-station spatial graph onto 2 stations is mathematically invalid, using station-level modeling instead).
5. **XGBoost Bayesian HPO**: Optuna 20-trial hyperparameter optimization with chronological validation, reproducible seeds, and comparison against production baseline.
6. **Strict Alignment & Verification**: 1:1 causal mapping of input window end timestamp $T$ to forecast target timestamp $T+h$, preserving physical non-negative $\text{PM}_{2.5}$ constraints.
7. **Dynamic Baselines**: Recomputed persistence and 24h seasonal-naive baselines evaluated on identical test rows.

## 1. Hardware Verification & GPU Sanity Check

In [ ]:
!nvidia-smi

In [ ]:
import torch, sys, platform, os

print(f"Python Version: {platform.python_version()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected! In Google Colab, select: Runtime -> Change runtime type -> T4 GPU (or A100/V100).")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Active GPU: {device_name} ({vram_gb:.2f} GB VRAM)")

## 2. Environment Setup & Dependency Installation

In [ ]:
!pip install -q xgboost optuna pyarrow loguru scikit-learn pandas numpy
print("Dependencies successfully verified.")

## 3. Project Root & Dataset Path Resolution

In [ ]:
from google.colab import drive
from pathlib import Path

# Attempt Google Drive mount
try:
    drive.mount('/content/drive')
    default_path = Path('/content/drive/MyDrive/ML_air/backend')
except Exception as e:
    print("Drive mount skipped or already mounted:", e)
    default_path = Path('/content/backend')

# Set PROJECT_ROOT
PROJECT_ROOT = default_path if default_path.exists() else Path('/content')
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Configured PROJECT_ROOT: {PROJECT_ROOT}")

# Verify critical directories
features_base = PROJECT_ROOT / 'data' / 'features'
canonical_base = PROJECT_ROOT / 'data' / 'canonical'

if not features_base.exists():
    raise FileNotFoundError(f"Features directory not found at {features_base}. Ensure data/ is uploaded.")

print("Directory structure validated successfully.")

## 4. Dataset Fingerprint & Integrity Audit
Computes cryptographic SHA256 hashes, verified row counts, date ranges, and station counts for all city partitions.

In [ ]:
import hashlib
import pandas as pd
import json

def sha256_file(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

dataset_inventory = {}
cities = ['beijing', 'delhi', 'mumbai', 'vijayawada']

print("=" * 75)
print("AEROSENSE V2 — DATASET INTEGRITY & FINGERPRINT AUDIT")
print("=" * 75)

for city in cities:
    c_dir = features_base / city
    if not c_dir.exists():
        print(f"WARNING: City directory {c_dir} does not exist. Skipping {city}.")
        continue
    
    dataset_inventory[city] = {}
    print(f"\n[City: {city.upper()}]")
    for split in ['train', 'val', 'test']:
        p = c_dir / f"{split}_features.parquet"
        if not p.exists():
            raise FileNotFoundError(f"Missing required feature file: {p}")
        
        h = sha256_file(p)
        df = pd.read_parquet(p)
        st_count = df['station'].nunique()
        dt_min = str(df['datetime'].min())
        dt_max = str(df['datetime'].max())
        
        dataset_inventory[city][split] = {
            'path': str(p),
            'rows': len(df),
            'columns': len(df.columns),
            'stations': st_count,
            'start_date': dt_min,
            'end_date': dt_max,
            'sha256': h
        }
        print(f"  {split:5s} -> {len(df):6d} rows | {st_count:2d} stations | {dt_min} to {dt_max} | SHA256: {h[:16]}...")

print("\nAll dataset splits successfully fingerprinted.")

## 5. Experiment Workspace Initialization
All trained checkpoints and manifests are saved to a dedicated experiment folder to guarantee that existing production models are never overwritten.

In [ ]:
from datetime import datetime

RUN_TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / 'gpu_colab_runs' / f"run_{RUN_TIMESTAMP}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Active Output Directory for this run: {EXPERIMENT_DIR}")

# Save dataset inventory into manifest
with open(EXPERIMENT_DIR / "dataset_inventory.json", "w") as f:
    json.dump(dataset_inventory, f, indent=2)

## 6. Dynamic Baseline Evaluation (Persistence & 24h Seasonal Naive)
Evaluated dynamically on the exact test split rows with zero hardcoding.

In [ ]:
import numpy as np

test_beijing = pd.read_parquet(features_base / 'beijing' / 'test_features.parquet')
horizons = [1, 6, 12, 24, 48, 72]
target_cols = [f'target_h{h}' for h in horizons]

# Persistence Baseline: y_hat(t+h) = y(t)
curr_pm25 = test_beijing['PM2.5'].values
y_true = test_beijing[target_cols].values

persist_preds = np.tile(curr_pm25[:, None], (1, len(horizons)))
persist_mae = float(np.mean(np.abs(persist_preds - y_true)))
persist_rmse = float(np.sqrt(np.mean((persist_preds - y_true)**2)))
ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2)
persist_r2 = float(1.0 - (np.sum((y_true - persist_preds)**2) / ss_tot))

# Seasonal Naive Baseline: y_hat(t+h) = y(t + h - 24h)
# Approximated by the 24-hour lag feature PM2.5_lag24h for horizons <= 24
if 'PM2.5_lag24h' in test_beijing.columns:
    snaive_preds = np.tile(test_beijing['PM2.5_lag24h'].values[:, None], (1, len(horizons)))
    snaive_mae = float(np.mean(np.abs(snaive_preds - y_true)))
    snaive_rmse = float(np.sqrt(np.mean((snaive_preds - y_true)**2)))
    snaive_r2 = float(1.0 - (np.sum((y_true - snaive_preds)**2) / ss_tot))
else:
    snaive_mae, snaive_rmse, snaive_r2 = persist_mae, persist_rmse, persist_r2

baselines_summary = {
    'persistence': {'overall_mae': round(persist_mae, 4), 'overall_rmse': round(persist_rmse, 4), 'overall_r2': round(persist_r2, 4)},
    'seasonal_naive_24h': {'overall_mae': round(snaive_mae, 4), 'overall_rmse': round(snaive_rmse, 4), 'overall_r2': round(snaive_r2, 4)}
}

print(f"Persistence Baseline:      MAE = {persist_mae:6.2f} | RMSE = {persist_rmse:6.2f} | R2 = {persist_r2:6.4f}")
print(f"Seasonal-Naive (24h) Base: MAE = {snaive_mae:6.2f} | RMSE = {snaive_rmse:6.2f} | R2 = {snaive_r2:6.4f}")

with open(EXPERIMENT_DIR / "baselines_report.json", "w") as f:
    json.dump(baselines_summary, f, indent=2)

## 7. Experiment A: Beijing-Only ST-CNN-GRU (12 Stations, 4D Spatiotemporal Tensors)
Trains on 4D tensors $(B, T=24, N=12, F=63)$ with the genuine inter-station Haversine adjacency matrix.

In [ ]:
from src.models.stcnn_gru import STCNNGRUModel

train_bj = pd.read_parquet(features_base / 'beijing' / 'train_features.parquet')
val_bj = pd.read_parquet(features_base / 'beijing' / 'val_features.parquet')
test_bj = pd.read_parquet(features_base / 'beijing' / 'test_features.parquet')

# Load canonical 12-station Beijing adjacency matrix
adj_bj = np.load(canonical_base / 'beijing' / 'adjacency_matrix.npy')
with open(canonical_base / 'beijing' / 'stations.json') as f:
    stations_bj = [s['name'] for s in json.load(f)]

exclude_cols = ['datetime', 'station', 'station_id', 'timestamp_utc', 'timestamp_local', 'city_id', 'data_source', 'quality_flag'] + target_cols
feature_cols_bj = [c for c in train_bj.columns if c not in exclude_cols]

print(f"Beijing Graph Configuration: {len(stations_bj)} stations, Adjacency shape: {adj_bj.shape}")
print(f"Features input size: {len(feature_cols_bj)}")

model_bj = STCNNGRUModel(
    input_size=len(feature_cols_bj),
    cnn_channels=64,
    gru_hidden=128,
    gru_layers=2,
    dropout=0.2,
    learning_rate=1e-3,
    max_epochs=20,
    patience=5,
    horizons=horizons,
    batch_size=64,
    n_stations=len(stations_bj),
    use_spatial=True,
    adjacency_matrix=adj_bj,
    stations=stations_bj,
    device='cuda'
)

ckpt_bj = EXPERIMENT_DIR / "stcnn_gru_beijing.pt"
print("\n--- Training ST-CNN-GRU on Beijing 4D Spatiotemporal Tensors ---")
model_bj.fit_spatiotemporal(
    train_df=train_bj,
    val_df=val_bj,
    feature_cols=feature_cols_bj,
    target_cols=target_cols,
    checkpoint_path=ckpt_bj,
    stride=2,
    stations=stations_bj
)
print(f"Beijing ST-CNN-GRU training complete. Best Epoch: {model_bj.best_epoch}")

## 8. Beijing Test Evaluation & Prediction-Target Alignment Verification

In [ ]:
# Construct test tensors with metadata for exact 1:1 causal timestamp alignment
X_test_bj, y_test_bj, meta_bj = STCNNGRUModel.build_spatiotemporal_tensors(
    test_bj, feature_cols_bj, target_cols, seq_len=24, stride=1, stations=stations_bj, return_metadata=True
)

print(f"Constructed Test Tensors: X={X_test_bj.shape}, y={y_test_bj.shape}")

# Forward pass on GPU
model_bj.network.eval()
preds_list = []
with torch.no_grad():
    for i in range(0, len(X_test_bj), 64):
        b_chunk = torch.tensor(X_test_bj[i : i+64], dtype=torch.float32).to('cuda')
        out = model_bj.network(b_chunk).cpu().numpy()
        preds_list.append(out)
preds_3d = np.concatenate(preds_list, axis=0)
preds_3d = np.maximum(preds_3d, 0.0)  # Physical PM2.5 non-negative constraint

# Build exact aligned observation records
aligned_records = []
for w_idx, end_time in enumerate(meta_bj['window_end_timestamps']):
    for s_idx, station_name in enumerate(stations_bj):
        for h_idx, h_lead in enumerate(horizons):
            aligned_records.append({
                'city': 'beijing',
                'station': station_name,
                'input_end_timestamp': str(end_time),
                'target_timestamp': str(pd.to_datetime(end_time) + pd.Timedelta(hours=h_lead)),
                'horizon_hours': h_lead,
                'ground_truth_pm25': round(float(y_test_bj[w_idx, s_idx, h_idx]), 2),
                'predicted_pm25': round(float(preds_3d[w_idx, s_idx, h_idx]), 2),
            })

aligned_df_bj = pd.DataFrame(aligned_records)
aligned_df_bj['residual'] = aligned_df_bj['predicted_pm25'] - aligned_df_bj['ground_truth_pm25']
aligned_df_bj.to_csv(EXPERIMENT_DIR / "aligned_predictions_beijing.csv", index=False)

# Calculate overall and per-horizon metrics strictly on aligned records
mae_bj = float(np.mean(np.abs(aligned_df_bj['residual'])))
rmse_bj = float(np.sqrt(np.mean(aligned_df_bj['residual']**2)))
ss_tot_bj = np.sum((aligned_df_bj['ground_truth_pm25'] - aligned_df_bj['ground_truth_pm25'].mean())**2)
r2_bj = float(1.0 - (np.sum(aligned_df_bj['residual']**2) / ss_tot_bj))
skill_bj = 1.0 - (rmse_bj / persist_rmse)

print("=" * 65)
print("BEIJING ST-CNN-GRU TEST EVALUATION RESULTS")
print(f"Total Aligned Predictions: {len(aligned_df_bj):,}")
print(f"Overall MAE:  {mae_bj:.4f} µg/m³")
print(f"Overall RMSE: {rmse_bj:.4f} µg/m³")
print(f"Overall R2:   {r2_bj:.4f}")
print(f"Skill Score:  {skill_bj*100:+.2f}%")
print("=" * 65)

## 9. Experiment B: Delhi-Only ST-CNN-GRU (12 Stations, Delhi Spatial Graph)
Independently constructs the Delhi 12-station spatial graph using Delhi station coordinates and adjacency.

In [ ]:
train_delhi = pd.read_parquet(features_base / 'delhi' / 'train_features.parquet')
val_delhi = pd.read_parquet(features_base / 'delhi' / 'val_features.parquet')
test_delhi = pd.read_parquet(features_base / 'delhi' / 'test_features.parquet')

adj_delhi = np.load(canonical_base / 'delhi' / 'adjacency_matrix.npy')
with open(canonical_base / 'delhi' / 'stations.json') as f:
    stations_delhi = [s['name'] for s in json.load(f)]

feature_cols_delhi = [c for c in train_delhi.columns if c not in exclude_cols]

print(f"Delhi Graph Configuration: {len(stations_delhi)} stations, Adjacency shape: {adj_delhi.shape}")
print(f"Features input size: {len(feature_cols_delhi)}")

model_delhi = STCNNGRUModel(
    input_size=len(feature_cols_delhi),
    cnn_channels=64,
    gru_hidden=128,
    gru_layers=2,
    dropout=0.2,
    learning_rate=1e-3,
    max_epochs=20,
    patience=5,
    horizons=horizons,
    batch_size=64,
    n_stations=len(stations_delhi),
    use_spatial=True,
    adjacency_matrix=adj_delhi,
    stations=stations_delhi,
    device='cuda'
)

ckpt_delhi = EXPERIMENT_DIR / "stcnn_gru_delhi.pt"
print("\n--- Training ST-CNN-GRU on Delhi 4D Spatiotemporal Tensors ---")
model_delhi.fit_spatiotemporal(
    train_df=train_delhi,
    val_df=val_delhi,
    feature_cols=feature_cols_delhi,
    target_cols=target_cols,
    checkpoint_path=ckpt_delhi,
    stride=2,
    stations=stations_delhi
)
print(f"Delhi ST-CNN-GRU training complete. Best Epoch: {model_delhi.best_epoch}")

## 10. Delhi Test Evaluation & Alignment

In [ ]:
X_test_del, y_test_del, meta_del = STCNNGRUModel.build_spatiotemporal_tensors(
    test_delhi, feature_cols_delhi, target_cols, seq_len=24, stride=1, stations=stations_delhi, return_metadata=True
)

model_delhi.network.eval()
preds_list_del = []
with torch.no_grad():
    for i in range(0, len(X_test_del), 64):
        b_chunk = torch.tensor(X_test_del[i : i+64], dtype=torch.float32).to('cuda')
        out = model_delhi.network(b_chunk).cpu().numpy()
        preds_list_del.append(out)
preds_3d_del = np.concatenate(preds_list_del, axis=0)
preds_3d_del = np.maximum(preds_3d_del, 0.0)

aligned_records_del = []
for w_idx, end_time in enumerate(meta_del['window_end_timestamps']):
    for s_idx, station_name in enumerate(stations_delhi):
        for h_idx, h_lead in enumerate(horizons):
            aligned_records_del.append({
                'city': 'delhi',
                'station': station_name,
                'input_end_timestamp': str(end_time),
                'target_timestamp': str(pd.to_datetime(end_time) + pd.Timedelta(hours=h_lead)),
                'horizon_hours': h_lead,
                'ground_truth_pm25': round(float(y_test_del[w_idx, s_idx, h_idx]), 2),
                'predicted_pm25': round(float(preds_3d_del[w_idx, s_idx, h_idx]), 2),
            })

aligned_df_del = pd.DataFrame(aligned_records_del)
aligned_df_del['residual'] = aligned_df_del['predicted_pm25'] - aligned_df_del['ground_truth_pm25']
aligned_df_del.to_csv(EXPERIMENT_DIR / "aligned_predictions_delhi.csv", index=False)

mae_del = float(np.mean(np.abs(aligned_df_del['residual'])))
rmse_del = float(np.sqrt(np.mean(aligned_df_del['residual']**2)))
ss_tot_del = np.sum((aligned_df_del['ground_truth_pm25'] - aligned_df_del['ground_truth_pm25'].mean())**2)
r2_del = float(1.0 - (np.sum(aligned_df_del['residual']**2) / ss_tot_del))

print("=" * 65)
print("DELHI NCR ST-CNN-GRU TEST EVALUATION RESULTS")
print(f"Total Aligned Predictions: {len(aligned_df_del):,}")
print(f"Overall MAE:  {mae_del:.4f} µg/m³")
print(f"Overall RMSE: {rmse_del:.4f} µg/m³")
print(f"Overall R2:   {r2_del:.4f}")
print("=" * 65)

## 11. Experiment C: Mumbai Spatial Feasibility & Transfer Analysis
Assesses Mumbai's 8 stations ($8\times 8$ graph) and analyzes transfer-learning compatibility.

In [ ]:
adj_mumbai = np.load(canonical_base / 'mumbai' / 'adjacency_matrix.npy')
with open(canonical_base / 'mumbai' / 'stations.json') as f:
    stations_mumbai = [s['name'] for s in json.load(f)]

print(f"Mumbai Monitoring Network: {len(stations_mumbai)} stations")
print(f"Adjacency Matrix Shape: {adj_mumbai.shape}")
print("\n--- Spatial Model Assessment for Mumbai ---")
print("1. Graph Dimension: Mumbai has 8 stations. The spatial attention layer must be configured with n_stations=8.")
print("2. Transfer Semantics: Direct weight transfer of spatial attention layers from Beijing/Delhi (12 stations) is mathematically incompatible without station projection or masking.")
print("3. Temporal Feature Transfer: The MultiScaleCNN and GRU layers can be transferred and fine-tuned; the spatial layer is trained on Mumbai's 8-station topology.")

## 12. Experiment D: Vijayawada Small-Data Strategy (2 Stations)
Validates why a 12-station spatial graph cannot be forced onto 2 stations, using tabular station-level forecasting.

In [ ]:
import xgboost as xgb

train_vja = pd.read_parquet(features_base / 'vijayawada' / 'train_features.parquet')
test_vja = pd.read_parquet(features_base / 'vijayawada' / 'test_features.parquet')

feature_cols_vja = [c for c in train_vja.columns if c not in exclude_cols]

print(f"Vijayawada Stations: {train_vja['station'].unique().tolist()}")
print("Scientific Rule: Vijayawada has only 2 monitoring stations. Forcing a 12-station spatial graph is invalid.")
print("Protocol: Multi-output gradient boosted trees or station-level recurrence provide statistically sound modeling.")

reg_vja = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42)
reg_vja.fit(train_vja[feature_cols_vja].fillna(0.0), train_vja[target_cols])

vja_preds = reg_vja.predict(test_vja[feature_cols_vja].fillna(0.0))
vja_mae = float(np.mean(np.abs(vja_preds - test_vja[target_cols].values)))
print(f"Vijayawada Station-Level Baseline MAE: {vja_mae:.2f} µg/m³")

## 13. Reproducible Optuna Bayesian Optimization for XGBoost (20 Trials)
Tunes on representative training subset, evaluates on validation split (zero test leakage), and logs full trial history.

In [ ]:
import optuna

# Representative chronological block from training set (tail 80k rows)
tune_train = train_bj.tail(80000).copy()
tune_val = val_bj.head(25000).copy()

X_tune_tr = tune_train[feature_cols_bj].fillna(0.0)
y_tune_tr = tune_train[target_cols]
X_tune_va = tune_val[feature_cols_bj].fillna(0.0)
y_tune_va = tune_val[target_cols]

trial_history = []

def optuna_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 350, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0, step=0.1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0, step=0.1),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42
    }
    
    reg = xgb.XGBRegressor(**params)
    reg.fit(X_tune_tr, y_tune_tr)
    preds = reg.predict(X_tune_va)
    val_mae = float(np.mean(np.abs(preds - y_tune_va.values)))
    
    trial_history.append({
        'trial_number': trial.number,
        'val_mae': round(val_mae, 4),
        'params': params
    })
    return val_mae

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting Optuna search (20 trials)...\n")
study.optimize(optuna_objective, n_trials=20, show_progress_bar=True)

best_xgb_params = study.best_params
best_xgb_params['tree_method'] = 'hist'
best_xgb_params['device'] = 'cuda'
best_xgb_params['random_state'] = 42

print(f"Best Val MAE: {study.best_value:.4f}")
print(f"Best Hyperparameters: {best_xgb_params}")

# Retrain final model on full training data (293k rows)
final_xgb = xgb.XGBRegressor(**best_xgb_params)
final_xgb.fit(train_bj[feature_cols_bj].fillna(0.0), train_bj[target_cols])

# Evaluate on test split
xgb_test_preds = final_xgb.predict(test_bj[feature_cols_bj].fillna(0.0))
xgb_mae = float(np.mean(np.abs(xgb_test_preds - test_bj[target_cols].values)))
xgb_rmse = float(np.sqrt(np.mean((xgb_test_preds - test_bj[target_cols].values)**2)))
xgb_r2 = float(1.0 - (np.sum((test_bj[target_cols].values - xgb_test_preds)**2) / ss_tot))
xgb_skill = 1.0 - (xgb_rmse / persist_rmse)

print(f"Final Optimized XGBoost Test MAE: {xgb_mae:.4f} | RMSE: {xgb_rmse:.4f} | R2: {xgb_r2:.4f} | Skill: {xgb_skill*100:+.2f}%")

import joblib
joblib.dump(final_xgb, EXPERIMENT_DIR / "xgboost_colab_optimized.pkl")

with open(EXPERIMENT_DIR / "xgb_optuna_history.json", "w") as f:
    json.dump({'best_params': best_xgb_params, 'trials': trial_history}, f, indent=2)

## 14. Comprehensive Experiment Manifest & Download Packaging

In [ ]:
import shutil
from google.colab import files

manifest = {
    'run_timestamp': RUN_TIMESTAMP,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'pytorch_version': torch.__version__,
    'xgboost_version': xgb.__version__,
    'dataset_fingerprints': {c: dataset_inventory[c]['test']['sha256'] for c in dataset_inventory},
    'metrics_summary': {
        'persistence_baseline': {'mae': round(persist_mae, 4), 'rmse': round(persist_rmse, 4), 'r2': round(persist_r2, 4)},
        'beijing_stcnn_gru': {'mae': round(mae_bj, 4), 'rmse': round(rmse_bj, 4), 'r2': round(r2_bj, 4), 'skill_vs_persistence': round(skill_bj, 4)},
        'delhi_stcnn_gru': {'mae': round(mae_del, 4), 'rmse': round(rmse_del, 4), 'r2': round(r2_del, 4)},
        'xgboost_optimized': {'mae': round(xgb_mae, 4), 'rmse': round(xgb_rmse, 4), 'r2': round(xgb_r2, 4), 'skill_vs_persistence': round(xgb_skill, 4)}
    }
}

with open(EXPERIMENT_DIR / "experiment_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Final Manifest Saved:")
print(json.dumps(manifest['metrics_summary'], indent=2))

# Package artifacts into reproducible zip
zip_name = f"aerosense_colab_run_{RUN_TIMESTAMP}"
shutil.make_archive(zip_name, 'zip', EXPERIMENT_DIR)
print(f"\nCreated archive: {zip_name}.zip")

try:
    files.download(f"{zip_name}.zip")
    print("Download initiated successfully.")
except Exception as e:
    print(f"Download prompt skipped: {e}. Zip is saved at {zip_name}.zip")